# BarqTrain Colab: Benchmark Comparison

This notebook compares four inference setups on the same prompt and output length:
- Baseline Hugging Face / PyTorch
- BarqTrain patch only
- BarqTrain patch with CUDA backend enabled
- Unsloth AI with full-weight precision (not 4-bit)

Methodology in this version:
- `ninja` is installed explicitly for native CUDA builds.
- Unsloth is installed using its official auto-install flow.
- Each setup is loaded, benchmarked, then cleaned up before the next run.
- The benchmark helper measures average latency, median latency, tokens/sec, and peak VRAM.


In [ ]:
%%capture
import os, importlib.util, shutil
!apt-get -y install python3-dev curl build-essential >/dev/null
if not os.path.exists(os.path.expanduser('~/.cargo/bin/cargo')):
    !curl https://sh.rustup.rs -sSf | sh -s -- -y >/dev/null
os.environ['PATH'] = f"{os.path.expanduser('~/.cargo/bin')}:{os.environ['PATH']}"
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq         "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2         "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo"         "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
!uv pip install -qqq ninja packaging pandas datasets accelerate peft
if not os.path.isdir("/content/BarqTrain"):
    !git clone https://github.com/YASSERRMD/BarqTrain.git /content/BarqTrain
%cd /content/BarqTrain
!pip install -q -e . --no-build-isolation
!BARQTRAIN_BUILD_CUDA=1 pip install -q -e . --no-build-isolation
assert shutil.which("ninja"), "ninja is required for the benchmark"
assert importlib.util.find_spec("barqtrain_rs"), "barqtrain_rs did not build; do not trust the benchmark"
assert importlib.util.find_spec("barqtrain_cuda"), "barqtrain_cuda did not build; do not trust the benchmark"


In [ ]:
%cd /content/BarqTrain

import copy
import gc
import importlib
import os
import statistics
import time

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_MODEL_NAME = "Qwen/Qwen2-0.5B-Instruct"
UNSLOTH_MODEL_NAME = "unsloth/Qwen2-0.5B-Instruct"
PROMPT = "Explain why fused kernels can improve LLM training throughput."
NEW_TOKENS = 64
WARMUP_RUNS = 2
MEASURE_RUNS = 5

def pick_dtype():
    if DEVICE != "cuda":
        return torch.float32
    if torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16

DTYPE = pick_dtype()

print("Device:", DEVICE)
print("Base model:", BASE_MODEL_NAME)
print("Unsloth model:", UNSLOTH_MODEL_NAME)
print("DType:", DTYPE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Warning: CUDA not available. CUDA and VRAM comparisons will be limited.")

import os
os.environ['BARQTRAIN_REQUIRE_RUST'] = '1'


In [ ]:
def clear_memory():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()


def load_tokenizer(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def load_hf_model(model_name, dtype):
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        trust_remote_code=True,
    )
    if DEVICE == "cuda":
        model = model.cuda()
    return model


def build_generation_kwargs(model, max_new_tokens):
    generation_config = copy.deepcopy(getattr(model, "generation_config", None))
    if generation_config is not None:
        generation_config.do_sample = False
        for attr in ("temperature", "top_p", "top_k"):
            if hasattr(generation_config, attr):
                setattr(generation_config, attr, None)
        return {
            "max_new_tokens": max_new_tokens,
            "generation_config": generation_config,
        }

    return {
        "max_new_tokens": max_new_tokens,
        "do_sample": False,
        "temperature": None,
        "top_p": None,
        "top_k": None,
    }


def get_barqtrain_patch_fn(enable_cuda_backend):
    os.environ["BARQTRAIN_AUTO_BUILD"] = "1" if enable_cuda_backend else "0"
    os.environ["BARQTRAIN_VERBOSE_BUILD"] = "0"

    import barqtrain._ffi as ffi
    import barqtrain.ops as ops
    import barqtrain.lora as lora
    import barqtrain.patch_models as patch_models

    ffi = importlib.reload(ffi)
    importlib.reload(ops)
    importlib.reload(lora)
    patch_models = importlib.reload(patch_models)

    backend_status = {
        "rust_backend_loaded": ffi.load_rust_backend() is not None,
        "cuda_backend_loaded": False,
    }
    if enable_cuda_backend and DEVICE == "cuda":
        backend_status["cuda_backend_loaded"] = ffi.load_cuda_backend() is not None
        if not backend_status["cuda_backend_loaded"]:
            raise RuntimeError(
                "BarqTrain CUDA backend failed to load. Re-run the BARQTRAIN_BUILD_CUDA install cell before trusting this benchmark."
            )

    return patch_models.patch_model, backend_status


def benchmark_generation(
    label,
    model,
    tokenizer,
    prompt,
    new_tokens,
    warmup_runs=2,
    measure_runs=5,
    extra_metrics=None,
):
    model.eval()
    if hasattr(model, "config"):
        model.config.use_cache = True

    inputs = tokenizer(prompt, return_tensors="pt")
    if DEVICE == "cuda":
        inputs = {k: v.cuda() for k, v in inputs.items()}

    clear_memory()

    with torch.inference_mode():
        warmup_kwargs = build_generation_kwargs(model, min(16, new_tokens))
        for _ in range(warmup_runs):
            _ = model.generate(**inputs, **warmup_kwargs)

    latencies = []
    with torch.inference_mode():
        generate_kwargs = build_generation_kwargs(model, new_tokens)
        for _ in range(measure_runs):
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            outputs = model.generate(**inputs, **generate_kwargs)
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            latencies.append(time.perf_counter() - t0)

    generated_tokens = outputs.shape[-1] - inputs["input_ids"].shape[-1]
    avg_latency = sum(latencies) / len(latencies)
    median_latency = statistics.median(latencies)
    peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2) if DEVICE == "cuda" else 0.0

    result = {
        "setup": label,
        "dtype": str(DTYPE).replace("torch.", ""),
        "generated_tokens": generated_tokens,
        "avg_latency_s": avg_latency,
        "median_latency_s": median_latency,
        "tokens_per_sec": generated_tokens / avg_latency if avg_latency > 0 else 0.0,
        "peak_vram_mb": peak_vram_mb,
    }
    if extra_metrics:
        result.update(extra_metrics)
    return result


def run_case(label, model_builder, tokenizer_builder, extra_metrics=None):
    clear_memory()
    tokenizer = tokenizer_builder()
    model = model_builder()
    try:
        return benchmark_generation(
            label,
            model,
            tokenizer,
            PROMPT,
            NEW_TOKENS,
            WARMUP_RUNS,
            MEASURE_RUNS,
            extra_metrics=extra_metrics,
        )
    finally:
        del model
        del tokenizer
        clear_memory()


In [ ]:
results = []

results.append(
    run_case(
        "baseline_hf_full_weight",
        model_builder=lambda: load_hf_model(BASE_MODEL_NAME, DTYPE),
        tokenizer_builder=lambda: load_tokenizer(BASE_MODEL_NAME),
        extra_metrics={"cuda_backend_loaded": None, "rust_backend_loaded": None},
    )
)

patch_model_no_cuda, no_cuda_status = get_barqtrain_patch_fn(enable_cuda_backend=False)
results.append(
    run_case(
        "barqtrain_patch_python_fallback",
        model_builder=lambda: patch_model_no_cuda(load_hf_model(BASE_MODEL_NAME, DTYPE)),
        tokenizer_builder=lambda: load_tokenizer(BASE_MODEL_NAME),
        extra_metrics=no_cuda_status,
    )
)

patch_model_with_cuda, cuda_status = get_barqtrain_patch_fn(enable_cuda_backend=True)
results.append(
    run_case(
        "barqtrain_patch_cuda_backend",
        model_builder=lambda: patch_model_with_cuda(load_hf_model(BASE_MODEL_NAME, DTYPE)),
        tokenizer_builder=lambda: load_tokenizer(BASE_MODEL_NAME),
        extra_metrics=cuda_status,
    )
)

try:
    from unsloth import FastLanguageModel

    def build_unsloth_model():
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=UNSLOTH_MODEL_NAME,
            max_seq_length=1024,
            dtype=DTYPE,
            load_in_4bit=False,
        )
        FastLanguageModel.for_inference(model)
        if DEVICE == "cuda":
            model = model.cuda()
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        return model, tokenizer

    def unsloth_case():
        clear_memory()
        model, tokenizer = build_unsloth_model()
        try:
            return benchmark_generation(
                "unsloth_full_weight",
                model,
                tokenizer,
                PROMPT,
                NEW_TOKENS,
                WARMUP_RUNS,
                MEASURE_RUNS,
                extra_metrics={"cuda_backend_loaded": None, "rust_backend_loaded": None},
            )
        finally:
            del model
            del tokenizer
            clear_memory()

    results.append(unsloth_case())
except Exception as e:
    print("Unsloth benchmark skipped:", repr(e))


In [ ]:
df = pd.DataFrame(results)
df = df.sort_values("tokens_per_sec", ascending=False).reset_index(drop=True)
df


## Interpreting the benchmark

Treat the BarqTrain CUDA result as valid only when `cuda_backend_loaded == True`. If that flag is false, you measured PyTorch fallback code rather than BarqTrain's compiled kernels, and the comparison is not meaningful.

What each row means:
- `baseline_hf_full_weight`: plain Transformers generation.
- `barqtrain_patch_python_fallback`: BarqTrain patching without the native CUDA backend.
- `barqtrain_patch_cuda_backend`: BarqTrain patching with the compiled CUDA backend loaded.
- `unsloth_full_weight`: full-precision Unsloth inference path.

If Hugging Face baseline still wins against `barqtrain_patch_cuda_backend`, that is a real signal to investigate. If it only wins against the Python fallback row, the benchmark is behaving correctly.


In [ ]:
out_path = "/content/barqtrain_benchmark_results.csv"
df.to_csv(out_path, index=False)
print("Saved:", out_path)
